# Universe Returns — Phase 3 Validation

1. **Returns matrix** — monthly return heatmap, sorted by sector
2. **Sector cumulative returns** — equal-weighted by sector vs SPY / XHB / XLI
3. **Correlation heatmap** — monthly returns, clustered by sector
4. **IPO / history check** — flag tickers with data starting after 2018-01-01

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_URL = (
    f"postgresql://{os.environ.get('POSTGRES_USER','urbangrowth')}"
    f":{os.environ.get('POSTGRES_PASSWORD','changeme')}"
    f"@{os.environ.get('POSTGRES_HOST','localhost')}"
    f":{os.environ.get('POSTGRES_PORT','5432')}"
    f"/{os.environ.get('POSTGRES_DB','urbangrowth')}"
)
engine = create_engine(DB_URL)

DATA_ROOT    = Path(os.environ.get("URBANGROWTH_DATA_ROOT", "C:/urbangrowth_data"))
CACHE_PATH   = DATA_ROOT / "raw" / "markets" / "prices_daily_wide.parquet"
BACKTEST_START = pd.Timestamp("2018-01-01")

plt.rcParams.update({"figure.dpi": 120})
print("Setup complete")

---
## Load data

Load from the parquet cache (fast) and compute monthly returns. Also pull the universe
metadata from the DB for sector labels.

In [ ]:
# ── Universe metadata ──────────────────────────────────────────────────────────
import yaml

CONFIG_DIR = Path("../config")
with open(CONFIG_DIR / "universe.yaml") as f:
    _raw = yaml.safe_load(f)

universe_df = pd.DataFrame(_raw["universe"])[["symbol", "name", "sector_tag"]]

# Add benchmarks as their own group
benchmarks = pd.DataFrame([
    {"symbol": "SPY",  "name": "S&P 500 ETF",          "sector_tag": "_benchmark"},
    {"symbol": "XLI",  "name": "Industrial Select ETF",  "sector_tag": "_benchmark"},
    {"symbol": "KRE",  "name": "Regional Bank ETF",      "sector_tag": "_benchmark"},
    {"symbol": "URA",  "name": "Uranium ETF",             "sector_tag": "_benchmark"},
    {"symbol": "PAVE", "name": "Infrastructure ETF",     "sector_tag": "_benchmark"},
])
# XHB already in universe as control
meta = pd.concat([universe_df, benchmarks], ignore_index=True).set_index("symbol")

print(f"Universe: {len(universe_df)} tickers + {len(benchmarks)} extra benchmarks")
meta.groupby("sector_tag").size().to_frame("count")

In [ ]:
# ── Load price cache ───────────────────────────────────────────────────────────
if not CACHE_PATH.exists():
    raise FileNotFoundError(
        f"Price cache not found at {CACHE_PATH}\nRun: ug ingest markets"
    )

price_df = pd.read_parquet(CACHE_PATH)
price_df.index = pd.to_datetime(price_df.index)
print(f"Price cache: {price_df.shape}, {price_df.index.min().date()} – {price_df.index.max().date()}")

# Extract adj_close
level0 = price_df.columns.get_level_values(0).unique()
ac_key = "Adj Close" if "Adj Close" in level0 else "Close"
ac = price_df[ac_key].copy()
print(f"Adj Close: {ac.shape[1]} tickers")

In [ ]:
# ── Compute monthly returns ────────────────────────────────────────────────────
monthly_prices = ac.resample("ME").last()
monthly_ret    = monthly_prices.pct_change()

print(f"Monthly returns: {monthly_ret.shape}  ({monthly_ret.index.min().date()} – {monthly_ret.index.max().date()})")
monthly_ret.tail(3)

---
## 1. Universe Returns Matrix

Monthly return heatmap. Rows = tickers grouped by sector, columns = months.
Red = negative, green = positive (capped at ±15% for legibility).

In [ ]:
SECTOR_ORDER = [
    "homebuilder", "electrical_infra", "civil_construction", "materials",
    "equipment", "steel", "reit_industrial", "reit_residential",
    "regional_bank", "control", "_benchmark",
]

# Sort tickers by sector order, then alphabetically within sector
meta_sorted = (
    meta.assign(sector_order=meta["sector_tag"].map({s: i for i, s in enumerate(SECTOR_ORDER)}))
    .sort_values(["sector_order", "symbol"])
)
sorted_tickers = [s for s in meta_sorted.index if s in monthly_ret.columns]

# Restrict to backtest window for cleaner view
plot_ret = monthly_ret.loc[BACKTEST_START:, sorted_tickers].T

# Cap at ±15%
z = plot_ret.clip(-0.15, 0.15).values
x_labels = [d.strftime("%Y-%m") for d in plot_ret.columns]
y_labels = list(plot_ret.index)

fig = go.Figure(go.Heatmap(
    z=z,
    x=x_labels,
    y=y_labels,
    colorscale="RdYlGn",
    zmid=0,
    zmin=-0.15,
    zmax=0.15,
    colorbar=dict(title="Monthly Return", tickformat=".0%"),
    hovertemplate="%{y} | %{x}<br>Return: %{z:.1%}<extra></extra>",
))

fig.update_layout(
    title="Universe Monthly Returns (capped ±15%), grouped by sector",
    height=max(500, len(sorted_tickers) * 14),
    xaxis=dict(tickangle=-45, tickfont_size=9),
    yaxis=dict(tickfont_size=9),
    margin=dict(l=80, r=20, t=50, b=80),
)
fig.show()

---
## 2. Sector Cumulative Returns

Equal-weighted average monthly return per sector, compounded from `backtest_start`.
Benchmark lines: SPY (grey), XHB (blue dashed), XLI (orange dashed).

In [ ]:
universe_tickers = universe_df["symbol"].tolist()
plot_window = monthly_ret.loc[BACKTEST_START:]

# Equal-weighted mean per sector (exclude control + _benchmark from sector lines)
sector_lines: dict[str, pd.Series] = {}
for sector in SECTOR_ORDER[:-2]:  # skip control, _benchmark
    members = [
        s for s in meta[meta["sector_tag"] == sector].index
        if s in plot_window.columns
    ]
    if not members:
        continue
    sector_lines[sector] = plot_window[members].mean(axis=1)

# Compute cumulative returns (rebased to 1.0 at backtest_start)
def cumret(s: pd.Series) -> pd.Series:
    return (1 + s.fillna(0)).cumprod()

fig = go.Figure()

# Benchmark lines first (grey / dashed)
bench_style = {"SPY": ("gray", "solid"), "XHB": ("royalblue", "dash"), "XLI": ("darkorange", "dash")}
for bench, (color, dash) in bench_style.items():
    if bench not in plot_window.columns:
        continue
    cr = cumret(plot_window[bench])
    fig.add_trace(go.Scatter(
        x=cr.index, y=cr.values, mode="lines",
        name=bench, line=dict(color=color, dash=dash, width=2),
    ))

# Sector lines
palette = px.colors.qualitative.Plotly
for i, (sector, ret) in enumerate(sector_lines.items()):
    cr = cumret(ret)
    fig.add_trace(go.Scatter(
        x=cr.index, y=cr.values, mode="lines",
        name=sector, line=dict(color=palette[i % len(palette)], width=1.5),
    ))

fig.update_layout(
    title=f"Sector Cumulative Returns (equal-weighted, rebased {BACKTEST_START.date()}=1.0)",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
    hovermode="x unified",
    legend=dict(orientation="v", x=1.01),
    height=500,
)
fig.show()

In [ ]:
# ── Annualised stats per sector ────────────────────────────────────────────────
spy_monthly = plot_window["SPY"] if "SPY" in plot_window.columns else None

stats_rows = []
for sector, ret in sector_lines.items():
    ann_ret  = (1 + ret.mean()) ** 12 - 1
    ann_vol  = ret.std() * np.sqrt(12)
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else np.nan
    max_dd   = ((1 + ret.fillna(0)).cumprod() / (1 + ret.fillna(0)).cumprod().cummax() - 1).min()
    spy_beta = None
    if spy_monthly is not None:
        common = ret.dropna().align(spy_monthly.dropna(), join="inner")
        cov = np.cov(common[0], common[1])
        spy_beta = cov[0, 1] / cov[1, 1] if cov[1, 1] > 0 else np.nan
    stats_rows.append({
        "sector":      sector,
        "ann_ret":     f"{ann_ret:.1%}",
        "ann_vol":     f"{ann_vol:.1%}",
        "sharpe":      f"{sharpe:.2f}",
        "max_drawdown": f"{max_dd:.1%}",
        "spy_beta":    f"{spy_beta:.2f}" if spy_beta is not None else "n/a",
    })

pd.DataFrame(stats_rows)

---
## 3. Correlation Heatmap

Monthly return correlations for the full universe. Tickers sorted by sector.

In [ ]:
# Only universe tickers (exclude extra benchmarks that aren't in universe)
corr_tickers = [s for s in sorted_tickers if s in universe_tickers and s in plot_window.columns]
corr_data = plot_window[corr_tickers].dropna(how="all")

# Require at least 24 months of overlap for correlation to be meaningful
corr_matrix = corr_data.corr(min_periods=24)

fig = px.imshow(
    corr_matrix,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Monthly Return Correlations (universe, sorted by sector)",
    labels=dict(color="Correlation"),
    aspect="auto",
)
fig.update_layout(
    height=700, width=750,
    xaxis=dict(tickfont_size=8, tickangle=-60),
    yaxis=dict(tickfont_size=8),
)
fig.show()

# Print the 10 highest off-diagonal correlations
corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["ticker_a", "ticker_b", "correlation"]
print("Top 10 highest correlations:")
print(corr_pairs.nlargest(10, "correlation").to_string(index=False))

In [ ]:
# ── Within-sector vs cross-sector average correlations ────────────────────────
within, cross = [], []
for i, sa in enumerate(corr_tickers):
    for sb in corr_tickers[i+1:]:
        c = corr_matrix.loc[sa, sb]
        if pd.isna(c):
            continue
        if meta.loc[sa, "sector_tag"] == meta.loc[sb, "sector_tag"]:
            within.append(c)
        else:
            cross.append(c)

print(f"Within-sector avg correlation : {np.mean(within):.3f} (n={len(within)})")
print(f"Cross-sector avg correlation  : {np.mean(cross):.3f} (n={len(cross)})")

---
## 4. IPO / Insufficient History Check

Flags tickers whose price history starts after `backtest_start` (2018-01-01) or that have
more than 10% missing months since `backtest_start`.

In [ ]:
backtest_monthly = monthly_ret.loc[BACKTEST_START:]
total_months = len(backtest_monthly)

rows = []
for sym in universe_tickers:
    if sym not in ac.columns:
        rows.append({"symbol": sym, "first_date": None, "missing_months": total_months,
                     "missing_pct": 100.0, "flag": "NO DATA"})
        continue

    first = ac[sym].dropna().index.min()
    bt_slice = backtest_monthly[sym] if sym in backtest_monthly.columns else pd.Series(dtype=float)
    n_missing = int(bt_slice.isna().sum())
    missing_pct = n_missing / total_months * 100

    flag = ""
    if first > BACKTEST_START:
        flag = f"IPO {first.date()}"
    elif missing_pct > 10:
        flag = f"GAPS {missing_pct:.0f}%"

    sector = meta.loc[sym, "sector_tag"] if sym in meta.index else "unknown"
    rows.append({
        "symbol":         sym,
        "sector_tag":     sector,
        "first_date":     first.date() if first else None,
        "missing_months": n_missing,
        "missing_pct":    round(missing_pct, 1),
        "flag":           flag,
    })

history_df = pd.DataFrame(rows)
flagged = history_df[history_df["flag"] != ""]

print(f"Total universe tickers : {len(history_df)}")
print(f"Flagged (IPO or gaps)  : {len(flagged)}")
print()
print("Flagged tickers:")
print(flagged[["symbol", "sector_tag", "first_date", "missing_pct", "flag"]].to_string(index=False))

In [ ]:
# ── History start date per ticker ─────────────────────────────────────────────
fig = px.bar(
    history_df.sort_values("first_date"),
    x="symbol", y="missing_pct",
    color="sector_tag",
    title="Missing Months % Since 2018-01-01 by Ticker",
    labels={"missing_pct": "Missing Months (%)", "symbol": "Ticker"},
    category_orders={"symbol": history_df.sort_values("missing_pct", ascending=False)["symbol"].tolist()},
)
fig.add_hline(y=10, line_dash="dash", line_color="red", annotation_text="10% threshold")
fig.show()

---
## 5. DB Returns Table Spot-Check

In [ ]:
db_q = text("""
    SELECT
        symbol,
        COUNT(*)                            AS daily_rows,
        MIN(date)::date                     AS first_date,
        MAX(date)::date                     AS last_date,
        COUNT(*) FILTER (WHERE monthly_ret IS NOT NULL) AS monthly_rows,
        ROUND(AVG(daily_ret)::numeric * 252 * 100, 1)  AS ann_ret_pct,
        ROUND(STDDEV(daily_ret)::numeric * SQRT(252) * 100, 1) AS ann_vol_pct
    FROM returns
    WHERE symbol = ANY(:syms)
    GROUP BY symbol
    ORDER BY symbol
""")

sample_syms = ["DHI", "LEN", "PWR", "VMC", "NUE", "SPY", "XHB"]
with engine.connect() as conn:
    db_stats = pd.read_sql(db_q, conn, params={"syms": sample_syms})

if db_stats.empty:
    print("returns table is empty — run: ug ingest markets")
else:
    display(db_stats)

In [ ]:
# ── Excess return summary ──────────────────────────────────────────────────────
excess_q = text("""
    SELECT
        r.symbol,
        t.sector_tag,
        ROUND(AVG(r.excess_ret_spy)::numeric  * 252 * 100, 2) AS ann_alpha_vs_spy_bps,
        ROUND(AVG(r.excess_ret_sector)::numeric * 252 * 100, 2) AS ann_alpha_vs_sector_bps
    FROM returns r
    JOIN tickers t ON t.symbol = r.symbol
    WHERE r.date >= :start
    GROUP BY r.symbol, t.sector_tag
    ORDER BY t.sector_tag, ann_alpha_vs_spy_bps DESC
""")

with engine.connect() as conn:
    excess = pd.read_sql(excess_q, conn, params={"start": BACKTEST_START.date()})

if excess.empty:
    print("returns or tickers table empty — run: ug ingest markets")
else:
    fig = px.bar(
        excess,
        x="symbol", y="ann_alpha_vs_spy_bps",
        color="sector_tag",
        title="Annualised Daily Alpha vs SPY (bps) since 2018",
        labels={"ann_alpha_vs_spy_bps": "Alpha vs SPY (ann. bps)"},
        category_orders={"symbol": excess.sort_values("ann_alpha_vs_spy_bps", ascending=False)["symbol"].tolist()},
    )
    fig.add_hline(y=0, line_color="black", line_width=1)
    fig.show()